# Predicting O-Level `final_test` Scores: EDA and Model Comparison

**Goal:** understand which student factors are linked to the `final_test` score, then build a model that predicts it.

**Data:** `data/data.csv`, 15,900 students and 17 columns: background (siblings, gender, age), school details (CCA, direct admission, class make-up), and habits (study hours, attendance, sleep and wake times, tuition, transport).

**Contents**
1. Inspect the data
2. Correct the data
3. Drop rows with incomplete or impossible data
4. Hours of sleep (new feature) and sleep timing
5. How `final_test` is distributed, and the key factors
6. Correlation between variables
7. EDA summary
8. Modelling: baseline, linear models and random forest
9. Final model choice

The chosen model is built into the pipeline in `main.py` and `src/`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")   # one plot style for the whole notebook

In [ ]:
df = pd.read_csv('data/data.csv', index_col=[0])

## Inspect the data (before any cleaning)

A first look at the raw data: its size, the column types, missing values, the target column, and the values in each categorical column. Nothing is changed yet.

In [ ]:
### Size of the dataset and the first few rows
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head()

In [ ]:
### Which columns are numbers and which are text (categories)?
numeric_columns = df.select_dtypes(include='number').columns.tolist()
categorical_columns = df.select_dtypes(exclude='number').columns.tolist()

print('Target column (what we predict): final_test')
print('\nNumeric columns:', len(numeric_columns))
print(numeric_columns)
print('\nCategorical / text columns:', len(categorical_columns))
print(categorical_columns)
print()
df.info()

In [ ]:
### Missing values per column (only columns with at least one missing value)
missing = df.isna().sum().to_frame('missing rows')
missing['% of rows'] = (missing['missing rows'] / len(df) * 100).round(2)
missing[missing['missing rows'] > 0]

In [ ]:
### Summary of the numeric columns, including the target final_test
df.describe().T.round(2)

In [ ]:
### Is student_id unique? Are there duplicate rows?
print('Rows:', len(df))
print('Unique student_id values:', df['student_id'].nunique())
print('Rows that are an exact copy of another row:', df.duplicated().sum())

In [ ]:
### Profile the categorical columns: each unique value, its count and its share of rows
def unique_count(column):
    counts = df[column].value_counts(dropna=False).reset_index()   # dropna=False keeps the NaN row
    counts.columns = ['unique', 'count']
    counts['% of rows'] = (counts['count'] / len(df) * 100).round(2)
    return counts

# tuition, CCA and age need fixing, so they are profiled in "Correct the data" below
for column in ['gender', 'direct_admission', 'learning_style', 'mode_of_transport', 'bag_color']:
    print(column)
    display(unique_count(column))

### First observations

- **Size:** 15,900 rows and 17 columns. The target is **`final_test`**, a score from 32 to 100 (average about 67).
- **Column types:** 7 numeric and 10 text columns (including `student_id`). `sleep_time` and `wake_time` are stored as text (e.g. "22:30"), so they are turned into hours of sleep later.
- **Missing values:** `CCA` (3,829 rows, 24%), `attendance_rate` (778, 4.9%) and `final_test` (495, 3.1%). The `CCA` gaps are really the category "None", which pandas reads as missing (see "Correct the data").
- **Impossible values:** `age` has a minimum of -5, and some ages are 5 or 6 when all students are 15 or 16. These are fixed or dropped in the next steps.
- **`student_id` is not unique:** there are 15,000 unique IDs for 15,900 rows, and 139 rows are exact copies of another row. These duplicates are dropped in the drop step. Because the IDs are (almost) one per student, they carry no pattern a model can learn from, so `student_id` is not used as a feature.
- **Categorical columns with clean labels:** `gender` (50 / 50), `direct_admission` (70% No), `learning_style` (57% Auditory), `mode_of_transport` (40% public, 40% private, 20% walk) and `bag_color` (6 colours, about 17% each). None of them need fixing, and none of the categories is so rare that it needs grouping. Only `tuition` and `CCA` have messy labels.

## Correct the data

Before replacing anything, list every **unique** value with its **count**. This shows which values are messy and how many rows each fix affects.

- **tuition**: `Y` / `N` are short forms of `Yes` / `No`. They are only a few hundred rows, next to thousands of `Yes` / `No`.
- **CCA**: `ARTS`, `CLUBS`, `SPORTS`, `NONE` are the same categories in a different case. There is no `None` row at all, but there are about 3,800 `NaN` rows. That is because pandas reads the text "None" as missing. So `NaN` → `None` puts the category back, and its count is then similar to the other three.
- **age**: the students are 15 or 16. `5` and `6` look like the leading `1` was lost, so `5` → `15` and `6` → `16`. Negative ages can't be fixed and are dropped in the next step.

In [ ]:
### BEFORE correcting: unique values and their counts
print('tuition')
display(unique_count('tuition'))
print('CCA')
display(unique_count('CCA'))
print('age')
display(unique_count('age'))

In [ ]:
### Correct the data
df['tuition'] = df['tuition'].replace('Y', 'Yes').replace('N', 'No')
df['CCA'] = df['CCA'].str.capitalize().fillna('None')
df['age'] = df['age'].replace(5, 15).replace(6, 16)

In [ ]:
### AFTER correcting: each column now has only the expected values
print('tuition')
display(unique_count('tuition'))
print('CCA')
display(unique_count('CCA'))
print('age')
display(unique_count('age'))

## Drop rows with incomplete or impossible data

These rows are dropped instead of filled in:

- **`final_test` missing**: this is the target. A row without it can't be used to train or test the model, and filling it in would mean making up the answer.
- **`age` ≤ 0**: impossible values with no clear correction.
- **Duplicate rows**: rows that are an exact copy of another row (same `student_id` and same values in every column). They are the same student recorded twice, so only one copy is kept. Counting is done after the corrections above, so rows that only differed by `Y`/`Yes` or `ARTS`/`Arts` count as duplicates too.

**`attendance_rate` missing (about 5% of rows) is kept, not dropped.** These students look like everyone else: their average `final_test` is about the same as the rest (shown in the cell below). Keeping them gives the model about 5% more students, so they are kept and filled with the **median** attendance (95%). The median is used instead of the average because attendance is bunched near 100% with a tail of low values, which pulls the average down. The fill is done **after the train-test split**, using the median of the training rows only, so the test set doesn't leak in.

The cell below counts the rows dropped and shows what share of the dataset they are. It then checks that the mean of each column barely moves. Together, this shows that dropping them loses only a small part of the data and doesn't bias it.

In [ ]:
### Drop rows with incomplete data: how many rows, and does it change the data?
rows_before = len(df)
missing_final_test = df['final_test'].isna().sum()
bad_age = (df['age'] <= 0).sum()
duplicate_rows = df.duplicated().sum()   # rows that are an exact copy of an earlier row
missing_attendance = df['attendance_rate'].isna().sum()

print('Rows in dataset:', rows_before)
print('Rows with missing final_test:', missing_final_test, '(', round(missing_final_test / rows_before * 100, 2), '% ) -> drop')
print('Duplicate rows:', duplicate_rows, '(', round(duplicate_rows / rows_before * 100, 2), '% ) -> drop')
print('Rows with age <= 0:', bad_age, '(', round(bad_age / rows_before * 100, 2), '% ) -> drop')
print('Rows with missing attendance_rate:', missing_attendance, '(', round(missing_attendance / rows_before * 100, 2), '% ) -> keep, fill with median later')

mean_before = df.mean(numeric_only=True)

df = df.drop_duplicates()
df = df.dropna(subset=['final_test'])
df = df[df['age'] > 0]

mean_after = df.mean(numeric_only=True)

# Students with missing attendance score about the same as the rest, so filling (not dropping) them is safe
missing_att = df['attendance_rate'].isna()
print('Average final_test, attendance missing:', round(df[missing_att]['final_test'].mean(), 2),
      '| attendance present:', round(df[~missing_att]['final_test'].mean(), 2))

rows_dropped = rows_before - len(df)
print('\nRows after dropping:', len(df), '| dropped', rows_dropped, '(', round(rows_dropped / rows_before * 100, 2), '% of the data )')

# If the means barely move, the dropped rows were not different from the rest
compare = pd.DataFrame({'mean before': mean_before, 'mean after': mean_after})
compare['change %'] = (compare['mean after'] - compare['mean before']) / compare['mean before'] * 100
compare.drop(index='age').round(2)

## Hours of sleep (new feature) and sleep timing

`sleep_time` and `wake_time` are text, so they are turned into a number: **hours of sleep**. Bedtimes before midnight need care: 22:00 to 06:00 is 8 hours, not -16. Taking the difference modulo one day (24 hours) handles this.

In [ ]:
### New column: sleep_hours
sleep = pd.to_timedelta(df['sleep_time'] + ':00')   # "22:00" -> "22:00:00" -> 22 hours
wake = pd.to_timedelta(df['wake_time'] + ':00')     # "6:00"  -> 6 hours

# % 1 day wraps overnight sleep around midnight: 6:00 - 22:00 = -16 hrs -> +8 hrs
df['sleep_hours'] = ((wake - sleep) % pd.Timedelta(days=1)).dt.total_seconds() / 3600

# Check a few rows, including after-midnight bedtimes
check = pd.concat([df[df['sleep_time'] == '22:00'].head(2), df[df['sleep_time'] == '1:00'].head(2)])
check[['sleep_time', 'wake_time', 'sleep_hours']]

In [ ]:
### Raw sleep_time and wake_time: when do students go to bed and wake up?
bedtime_order = ['21:00', '21:30', '22:00', '22:30', '23:00', '23:30',
                 '0:00', '0:30', '1:00', '1:30', '2:00', '2:30', '3:00']   # in time order, not text order
bedtime_counts = df['sleep_time'].value_counts().reindex(bedtime_order)
wake_counts = df['wake_time'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [13, 5]})

bars = ax1.bar(bedtime_counts.index, bedtime_counts.values, color='#3b6fb6')
ax1.bar_label(bars, fontsize=9)
ax1.set_title('Bedtime (sleep_time)', loc='left')
ax1.set_xlabel('Bedtime')
ax1.set_ylabel('Number of students')

bars = ax2.bar(wake_counts.index, wake_counts.values, color='#3b6fb6')
ax2.bar_label(bars, fontsize=9)
ax2.set_title('Wake-up time (wake_time)', loc='left')
ax2.set_xlabel('Wake-up time')
ax2.set_ylabel('Number of students')

plt.tight_layout()
plt.show()

print('Share of students going to bed between 21:00 and 23:00:',
      round(bedtime_counts.loc['21:00':'23:00'].sum() / len(df) * 100, 1), '%')
print('Share of students waking at each time (%):')
print((wake_counts / len(df) * 100).round(1))

- **Bedtimes:** most students go to bed between 21:00 and 23:00, about 19% at each half-hour. Only about 5% go to bed later than 23:00, but some stay up as late as 03:00.
- **Wake-up times:** spread evenly from 05:00 to 07:00, about 20% at each half-hour. Nobody wakes later than 07:00.

So a later bedtime can't be made up by sleeping in. The chart below shows what that does to hours of sleep.

### When students sleep vs how long they sleep

Left: each bar is one bedtime, running from the bedtime to the **average wake time** of students with that bedtime. Right: the average hours of sleep for each bedtime.

The question: do students who go to bed earlier or later get more or less sleep?

In [ ]:
### Sleep window (bedtime -> wake time) next to hours of sleep
sleep_df = df[['sleep_time', 'wake_time', 'sleep_hours']].copy()

# Turn clock times into numbers so they can go on an axis, e.g. "22:30" -> 22.5
sleep_df['bed_hour'] = pd.to_timedelta(sleep_df['sleep_time'] + ':00').dt.total_seconds() / 3600
sleep_df['wake_hour'] = pd.to_timedelta(sleep_df['wake_time'] + ':00').dt.total_seconds() / 3600

# Put the night on one line: after-midnight bedtimes (e.g. 1:00) become 25, wake times (all in the morning) get +24
sleep_df.loc[sleep_df['bed_hour'] < 12, 'bed_hour'] = sleep_df['bed_hour'] + 24
sleep_df['wake_hour'] = sleep_df['wake_hour'] + 24

# One row per bedtime
summary = sleep_df.groupby('bed_hour').agg(
    bedtime=('sleep_time', 'first'),
    students=('sleep_time', 'count'),
    avg_wake_hour=('wake_hour', 'mean'),
    avg_sleep_hours=('sleep_hours', 'mean'),
).reset_index()

# Blue = 7+ hours of sleep on average, orange = under 7 hours
colors = ['#3b6fb6' if h >= 7 else '#d1603d' for h in summary['avg_sleep_hours']]
rows = range(len(summary))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), sharey=True, gridspec_kw={'width_ratios': [3, 2]})

# Left chart: a bar from bedtime to average wake time
ax1.barh(rows, summary['avg_wake_hour'] - summary['bed_hour'], left=summary['bed_hour'], color=colors, height=0.6)
for row, wake, n in zip(rows, summary['avg_wake_hour'], summary['students']):
    ax1.text(wake + 0.15, row, f'n={n:,}', va='center', fontsize=9)
ax1.set_xticks(range(21, 33),
               ['21:00', '22:00', '23:00', '00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00'])
ax1.set_xlim(20.5, 33)
ax1.set_yticks(rows, summary['bedtime'])
ax1.invert_yaxis()   # earliest bedtime at the top
ax1.set_xlabel('Time of night')
ax1.set_ylabel('Bedtime')
ax1.set_title('Sleep window: bedtime -> average wake time', loc='left')

# Right chart: average hours of sleep for each bedtime
ax2.plot(summary['avg_sleep_hours'], rows, color='#555555', linewidth=2)
ax2.scatter(summary['avg_sleep_hours'], rows, color=colors, s=80, zorder=2)
for row, hours in zip(rows, summary['avg_sleep_hours']):
    ax2.text(hours + 0.15, row, f'{hours:.1f} hrs', va='center', fontsize=9)
ax2.axvline(7, color='#888888', linestyle='--', linewidth=1)   # 7-hour line
ax2.set_xlim(2, 10)
ax2.set_xlabel('Average hours of sleep (dashed line = 7 hrs)')
ax2.set_title('Hours of sleep by bedtime', loc='left')

fig.suptitle('Later bedtimes mean less sleep: nobody wakes after 07:00   (blue = 7+ hrs, orange = under 7 hrs)',
             fontsize=13, x=0.01, ha='left')
plt.tight_layout()
plt.show()

summary[['bedtime', 'students', 'avg_sleep_hours']].round(2)

## How `final_test` is distributed, and the key factors

In [ ]:
### Distribution of the target, final_test
plt.figure(figsize=(10, 6))
sns.histplot(df['final_test'], kde=True)
plt.title('Distribution of final_test')
plt.xlabel('final_test score')
plt.ylabel('Number of students')
plt.show()

print('Mean:', round(df['final_test'].mean(), 1), '| Median:', df['final_test'].median(),
      '| Min:', df['final_test'].min(), '| Max:', df['final_test'].max(),
      '| Skew:', round(df['final_test'].skew(), 2))

- Scores range from 32 to 100, with a mean of about 67 and a median of 68.
- The shape is close to symmetric (skew about 0.06), so the target doesn't need a log or other transformation, and there are no extreme outliers: every score is a possible test mark.
- The spikes are because scores are whole numbers grouped into bins. They are not a data problem.

In [ ]:
### Average final_test score by key factors
sns.set_style("whitegrid")
BAR = '#3b6fb6'                       # one colour: every panel is a single series
overall_mean = df['final_test'].mean()

def group_stats(col):
    g = df.groupby(col)['final_test']
    out = g.agg(['mean', 'count', 'sem']).reset_index()
    out['ci'] = 1.96 * out['sem']     # 95% confidence interval
    return out

def finish(ax, title, xlabel, col):
    r = df[col].replace({'Yes': 1, 'No': 0}).astype(float).corr(df['final_test'])
    ax.axhline(overall_mean, color='#888888', linestyle='--', linewidth=1,
               label=f'Overall average ({overall_mean:.0f})')
    ax.legend(loc='upper left', fontsize=9, frameon=False)
    ax.set_title(f'{title}   (Pearson r = {r:.2f})', loc='left', fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Average final_test score')
    ax.set_ylim(0, 100)
    ax.grid(axis='x', visible=False)

def bar_panel(ax, col, order=None, labels=None):
    s = group_stats(col)
    if order is not None:
        s = s.set_index(col).loc[order].reset_index()
    names = labels or [str(v) for v in s[col]]
    x = np.arange(len(s))
    ax.bar(x, s['mean'], yerr=s['ci'], color=BAR, width=0.6, capsize=4,
           error_kw={'ecolor': '#333333', 'linewidth': 1})
    for xi, m in zip(x, s['mean']):
        ax.text(xi, m + 3, f'{m:.1f}', ha='center', fontsize=10, color='#222222')
    ax.set_xticks(x, [f'{n}\n(n={c:,})' for n, c in zip(names, s['count'])])

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Study hours per week (line: 0-20 hours is a continuous scale)
s = group_stats('hours_per_week')
ax = axes[0, 0]
ax.fill_between(s['hours_per_week'], s['mean'] - s['ci'], s['mean'] + s['ci'], color=BAR, alpha=0.2, linewidth=0)
ax.plot(s['hours_per_week'], s['mean'], color=BAR, linewidth=2, marker='o', markersize=6)
ax.axvspan(4.5, 10.5, color='#2e7d32', alpha=0.07)
ax.text(7.5, 92, 'sweet spot: 5-10 hrs', ha='center', fontsize=9, color='#2e7d32')
ax.set_xticks(range(0, 21, 2))
finish(ax, '1. Study hours per week', 'Hours studied per week', 'hours_per_week')

# 2. Sleep hours
bar_panel(axes[0, 1], 'sleep_hours', labels=None)
axes[0, 1].set_xticks(axes[0, 1].get_xticks(),
                      [t.get_text().replace('.0', ' hrs') for t in axes[0, 1].get_xticklabels()])
finish(axes[0, 1], '2. Hours of sleep', 'Hours of sleep per night', 'sleep_hours')

# 3. Number of siblings
bar_panel(axes[1, 0], 'number_of_siblings')
finish(axes[1, 0], '3. Number of siblings', 'Number of siblings', 'number_of_siblings')

# 4. Tuition
bar_panel(axes[1, 1], 'tuition', order=['No', 'Yes'], labels=['No tuition', 'Has tuition'])
finish(axes[1, 1], '4. Tuition', '', 'tuition')

fig.suptitle('Average final_test score by factor  (error bars / band = 95% confidence interval)',
             fontsize=14, x=0.01, ha='left')
plt.tight_layout()
plt.show()

- **Study hours:** the clearest non-linear pattern. Students who study 5-10 hours a week average about 72, against about 46 for 0-4 hours and about 62 for 11+ hours. Pearson r is only -0.15, because a straight line can't describe a peak. This is why `study_band` is created for the model.
- **Sleep:** students with 7-8 hours of sleep score far higher (61-68) than those with 4-6 hours (44-46). The big drop is below 7 hours, which is why `enough_sleep` is created.
- **Siblings:** students with no siblings average 76, against 62-64 for one or two siblings.
- **Tuition:** students with tuition average 70.5, against 62.8 without.
- The 95% confidence intervals are narrow because each group is large, so these gaps are not chance.

## Correlation between variables

Three views:
1. **Pearson correlation heatmap** – numeric variables plus Yes/No and Male/Female columns encoded as 0/1.
2. **Correlation with `final_test`** – the same numbers as a sorted bar chart, so the strongest predictors stand out.
3. **Association matrix across *all* variables** – handles multi-category text columns (CCA, learning_style, transport, bag_color) too:
   - numeric ↔ numeric: |Pearson r|
   - categorical ↔ numeric: correlation ratio η (eta)
   - categorical ↔ categorical: Cramér's V

   All three run from 0 (no relationship) to 1 (perfect relationship), so they can share one colour scale.

In [ ]:
### Build a correlation-ready dataframe
corr_df = df.drop(columns=['student_id', 'sleep_time', 'wake_time'], errors='ignore').copy()

# Bedtime as a number (hours); times after midnight count as 24+ so 1:00 -> 25 (later than 23:00)
bed = pd.to_timedelta(df['sleep_time'] + ':00').dt.total_seconds() / 3600
corr_df['bedtime_hr'] = np.where(bed < 12, bed + 24, bed)

# Encode two-level categories as 0/1 so they can join the Pearson heatmap
corr_df['direct_admission'] = corr_df['direct_admission'].map({'No': 0, 'Yes': 1})
corr_df['tuition'] = corr_df['tuition'].map({'No': 0, 'Yes': 1})
corr_df['gender_male'] = corr_df.pop('gender').map({'Female': 0, 'Male': 1})

cat_cols = ['CCA', 'learning_style', 'mode_of_transport', 'bag_color']
num_cols = [c for c in corr_df.columns if c not in cat_cols]

# Put the target first so it is the top row / left column
num_cols = ['final_test'] + [c for c in num_cols if c != 'final_test']
corr_df[num_cols].describe().T

In [ ]:
### Chart 1: Pearson correlation heatmap (numeric + binary variables)
corr = corr_df[num_cols].corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # hide the duplicate upper triangle

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', annot_kws={'size': 9},
    cmap='RdBu_r', vmin=-1, vmax=1, center=0,           # diverging: blue = negative, red = positive, white = none
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.7, 'label': 'Pearson r'}
)
plt.title('Pearson correlation between numeric variables', fontsize=14, loc='left')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
### Chart 2: which variables move with final_test?
target_corr = corr['final_test'].drop('final_test').sort_values()

colors = np.where(target_corr < 0, '#2166ac', '#b2182b')   # same blue/red meaning as the heatmap

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(target_corr.index, target_corr.values, color=colors, height=0.6)
ax.axvline(0, color='#555555', linewidth=1)
for bar, v in zip(bars, target_corr.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), bar.get_y() + bar.get_height() / 2,
            f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9, color='#333333')
ax.set_xlim(-1, 1)
ax.set_xlabel('Pearson r with final_test  (blue = lower score, red = higher score)')
ax.set_title('Correlation of each variable with final_test', fontsize=14, loc='left')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
### Chart 3: association between ALL variables (numeric + categorical)

def cramers_v(x, y):
    """Cramér's V (bias-corrected) for two categorical series, 0..1."""
    table = pd.crosstab(x, y).values
    n = table.sum()
    if n == 0 or min(table.shape) < 2:
        return np.nan
    expected = table.sum(1, keepdims=True) * table.sum(0, keepdims=True) / n
    chi2 = ((table - expected) ** 2 / expected).sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2corr / min(kcorr - 1, rcorr - 1))

def correlation_ratio(categories, values):
    """Correlation ratio eta: how much of a numeric variable's variance is explained by a category, 0..1."""
    d = pd.DataFrame({'c': categories, 'v': values}).dropna()
    if d['v'].var() == 0:
        return np.nan
    grand_mean = d['v'].mean()
    groups = d.groupby('c')['v']
    ss_between = (groups.count() * (groups.mean() - grand_mean) ** 2).sum()
    ss_total = ((d['v'] - grand_mean) ** 2).sum()
    return np.sqrt(ss_between / ss_total)

all_cols = num_cols + cat_cols
assoc = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

for a in all_cols:
    for b in all_cols:
        if a == b:
            assoc.loc[a, b] = 1.0
        elif a in cat_cols and b in cat_cols:
            assoc.loc[a, b] = cramers_v(corr_df[a], corr_df[b])
        elif a in cat_cols:
            assoc.loc[a, b] = correlation_ratio(corr_df[a], corr_df[b])
        elif b in cat_cols:
            assoc.loc[a, b] = correlation_ratio(corr_df[b], corr_df[a])
        else:
            assoc.loc[a, b] = abs(corr_df[a].corr(corr_df[b]))

mask = np.triu(np.ones_like(assoc, dtype=bool), k=1)

plt.figure(figsize=(14, 12))
sns.heatmap(
    assoc, mask=mask, annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='Blues', vmin=0, vmax=1,                         # sequential: darker = stronger relationship
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.7, 'label': 'Strength of association (0 = none, 1 = perfect)'}
)
plt.title("Association between all variables\n|Pearson r| (num-num) · eta (cat-num) · Cramér's V (cat-cat)",
          fontsize=14, loc='left')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Ranked list of what relates most to final_test
assoc['final_test'].drop('final_test').sort_values(ascending=False).round(3).to_frame('association_with_final_test')

- **Strongest links with `final_test`:** CCA (eta 0.40; students with no CCA average about 77, against 64 for the rest), number of siblings (r -0.36), attendance rate (r 0.35), hours of sleep (r 0.33), learning style (visual learners average about 72, auditory 64), tuition (r 0.27) and direct admission (r 0.24).
- **Class make-up:** `n_male` and `n_female` are each only weakly linked (r about -0.15 and -0.17), but their sum, class size, is much stronger (r -0.50). That's why `class_size` is created.
- **No relationship:** gender, age, mode of transport and bag colour (all 0.02 or below). These are left out of the model.
- **Bedtime vs hours of sleep:** they overlap (a later bedtime means less sleep), so only `sleep_hours` is used.

## EDA summary

**Data quality issues and fixes**

| Issue found | Fix |
|---|---|
| `tuition` uses `Y`/`N` as well as `Yes`/`No` | Mapped to `Yes`/`No` |
| `CCA` has mixed casing, and "None" is read as missing | Capitalised; missing filled with `None` |
| `age` has typos 5 and 6, and impossible values (-5, -4) | 5 → 15, 6 → 16; 5 rows with age ≤ 0 dropped |
| 139 duplicate rows | Dropped |
| 495 rows with no `final_test` (the target) | Dropped |
| 778 rows with no `attendance_rate` | Kept; filled with the training median (95) after the split |

In total, 638 rows (4%) are dropped, leaving 15,262. Column averages barely change.

**What this means for the model**

- **Use:** class size (created from `n_male` + `n_female`), number of siblings, attendance rate, hours of sleep, CCA, learning style, tuition and direct admission.
- **Create features for non-linear patterns:** study-hour bands (peak at 5-10 hrs), a low-attendance flag (≤ 80%) and an enough-sleep flag (≥ 7 hrs).
- **Leave out:** gender, age, mode of transport, bag colour (no relationship) and `student_id` (an identifier, not a pattern).
- **Expect a non-linear model to do well:** several key relationships are thresholds or peaks, not straight lines.

## Modelling: predicting `final_test`

- **Split:** 80% training / 20% testing (`random_state=42`). GridSearchCV and RandomizedSearchCV tune with 5-fold cross-validation **inside the training set**, so the test set is only used once, for the final score.
- **Missing values:** `attendance_rate` is missing for about 5% of students. It is filled with the median of the training rows, right after the split.
- **Features:** chosen from the EDA. The weak or useless columns (gender, age, transport, bag colour, student ID) are dropped. Extra features are added for the patterns a straight line can't capture: class size, study-hour bands, and the low-attendance and too-little-sleep thresholds.
- **Models:** a **baseline** that always predicts the average score (any real model must beat it), then Linear Regression, Ridge, Lasso, Ridge and Lasso tuned with GridSearchCV and RandomizedSearchCV, and finally a **random forest** (a tree-based, non-linear model), untuned and tuned with RandomizedSearchCV.
- **Metrics:** MAE and RMSE (average error in marks, lower is better) and R² (share of the variation in scores explained, higher is better).

In [ ]:
### Feature engineering + 80/20 train-test split
from sklearn.model_selection import train_test_split

model_df = df.copy()

# Features suggested by the EDA (the patterns were thresholds / peaks, so give the linear models features that can express them)
model_df['class_size'] = model_df['n_male'] + model_df['n_female']                     # strongest single predictor (r = -0.50)
model_df['study_band'] = pd.cut(model_df['hours_per_week'], bins=[-1, 4, 10, 100],
                                labels=['0-4 hrs', '5-10 hrs', '11+ hrs']).astype(str)  # score peaks at 5-10 hrs
model_df['low_attendance'] = (model_df['attendance_rate'] <= 80).astype(int)           # big drop at <= 80% (missing -> 0)
model_df['enough_sleep'] = (model_df['sleep_hours'] >= 7).astype(int)                  # big drop below 7 hrs

numerical_features = ['class_size', 'attendance_rate', 'sleep_hours', 'low_attendance', 'enough_sleep']
nominal_features = ['CCA', 'learning_style', 'tuition', 'direct_admission', 'study_band', 'number_of_siblings']
# Left out on purpose (no relationship with final_test): gender, age, mode_of_transport, bag_color, student_id

X = model_df[numerical_features + nominal_features]
y = model_df['final_test']

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fill missing attendance_rate with the median of the TRAINING rows only (so the test set doesn't leak in)
attendance_median = X_train['attendance_rate'].median()
X_train = X_train.fillna({'attendance_rate': attendance_median})
X_test = X_test.fillna({'attendance_rate': attendance_median})
print('Missing attendance_rate filled with the training median:', attendance_median)

print(f'Training set: {X_train.shape[0]:,} rows ({X_train.shape[0] / len(X):.0%})')
print(f'Testing set:  {X_test.shape[0]:,} rows ({X_test.shape[0] / len(X):.0%})')

In [ ]:
### Preprocessing pipeline + helper to score a model on the test set
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('nom', OneHotEncoder(handle_unknown='ignore', drop='first'), nominal_features)
])

results = []   # collects the test-set score of every model for the final comparison

def evaluate(name, model):
    """Score a fitted model on the 20% test set and store the result."""
    y_pred = model.predict(X_test)
    scores = {
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': root_mean_squared_error(y_test, y_pred),
        'R²': r2_score(y_test, y_pred),
    }
    results.append(scores)
    print(f"{name:<28} MAE {scores['MAE']:.3f} | RMSE {scores['RMSE']:.3f} | R² {scores['R²']:.4f}")
    return scores

In [ ]:
### 1-3. Linear Regression, Ridge and Lasso (default settings, alpha = 1)
from sklearn.linear_model import LinearRegression, Ridge, Lasso

lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge(alpha=1))])
lasso_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(alpha=1))])

for name, pipe in [('Linear Regression', lr_pipeline),
                   ('Ridge (alpha=1)', ridge_pipeline),
                   ('Lasso (alpha=1)', lasso_pipeline)]:
    pipe.fit(X_train, y_train)
    evaluate(name, pipe)

In [ ]:
### 4. GridSearchCV: try every alpha in the list, using 5-fold cross-validation on the training set only
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'regressor__fit_intercept': [True, False]
}

ridge_grid_search = GridSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge())]),
                                 param_grid, cv=5, scoring='r2', n_jobs=-1)
ridge_grid_search.fit(X_train, y_train)

lasso_grid_search = GridSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(max_iter=10000))]),
                                 param_grid, cv=5, scoring='r2', n_jobs=-1)
lasso_grid_search.fit(X_train, y_train)

print('Best Ridge parameters:', ridge_grid_search.best_params_, f'(CV R² {ridge_grid_search.best_score_:.4f})')
print('Best Lasso parameters:', lasso_grid_search.best_params_, f'(CV R² {lasso_grid_search.best_score_:.4f})')
print()
evaluate('Ridge + GridSearchCV', ridge_grid_search.best_estimator_)
evaluate('Lasso + GridSearchCV', lasso_grid_search.best_estimator_)

In [ ]:
### 5. RandomizedSearchCV: sample 30 random alphas from a continuous range (0.0001 - 1000) instead of a fixed list
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

param_dist = {
    'regressor__alpha': loguniform(1e-4, 1e3),
    'regressor__fit_intercept': [True, False]
}

ridge_random_search = RandomizedSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge())]),
                                         param_distributions=param_dist, n_iter=30, cv=5, scoring='r2',
                                         random_state=42, n_jobs=-1)
ridge_random_search.fit(X_train, y_train)

lasso_random_search = RandomizedSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(max_iter=10000))]),
                                         param_distributions=param_dist, n_iter=30, cv=5, scoring='r2',
                                         random_state=42, n_jobs=-1)
lasso_random_search.fit(X_train, y_train)

print('Best Ridge parameters:', ridge_random_search.best_params_, f'(CV R² {ridge_random_search.best_score_:.4f})')
print('Best Lasso parameters:', lasso_random_search.best_params_, f'(CV R² {lasso_random_search.best_score_:.4f})')
print()
evaluate('Ridge + RandomizedSearchCV', ridge_random_search.best_estimator_)
evaluate('Lasso + RandomizedSearchCV', lasso_random_search.best_estimator_)

### 6. Baseline and a non-linear model (random forest)

All the models so far are straight-line (linear) models. Two more models put their scores in context:

- **Baseline:** always predicts the average `final_test` of the training set. It ignores every feature, so a useful model has to beat it by a wide margin.
- **Random forest:** many decision trees, each trained on a random sample of the data, whose predictions are averaged. Trees can pick up thresholds and interactions (for example "large class **and** low attendance") that a straight line can't.

The tuned forest uses the same settings as the pipeline in `src/config.yaml`.

In [ ]:
### 6. Baseline + random forest (untuned, then tuned with RandomizedSearchCV)
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

# Baseline: always predict the training-set average
baseline_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', DummyRegressor(strategy='mean'))])
baseline_pipeline.fit(X_train, y_train)
evaluate('Baseline (predict the mean)', baseline_pipeline)

# Random forest with default settings
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42))])
rf_pipeline.fit(X_train, y_train)
evaluate('Random forest (default)', rf_pipeline)

# Random forest tuned with RandomizedSearchCV: try 20 random combinations, 5-fold CV on the training set only
rf_params = {
    'regressor__n_estimators': [100, 200, 300],        # number of trees
    'regressor__max_depth': [None, 10, 15, 20],        # how deep each tree can grow
    'regressor__min_samples_leaf': [1, 2, 5, 10],      # smallest group of students a leaf can hold
    'regressor__max_features': ['sqrt', 0.5, 1.0],     # share of features each split can look at
}
rf_random_search = RandomizedSearchCV(rf_pipeline, param_distributions=rf_params, n_iter=20, cv=5,
                                      scoring='r2', random_state=42, n_jobs=-1)
rf_random_search.fit(X_train, y_train)

print('\nBest random forest parameters:', rf_random_search.best_params_, f'(CV R² {rf_random_search.best_score_:.4f})')
evaluate('Random forest + RandomizedSearchCV', rf_random_search.best_estimator_)

In [ ]:
### Is the tuned forest stable and not overfitting?
best_rf = rf_random_search.best_estimator_

# 1. Train vs test: a much better train score would mean overfitting
train_rmse = root_mean_squared_error(y_train, best_rf.predict(X_train))
test_rmse = root_mean_squared_error(y_test, best_rf.predict(X_test))
print(f'Train RMSE {train_rmse:.3f} | Test RMSE {test_rmse:.3f} | gap {test_rmse - train_rmse:.3f} marks')

# 2. Fold-to-fold variation: R² of the best settings on each of the 5 CV folds
best = rf_random_search.best_index_
fold_scores = [rf_random_search.cv_results_[f'split{i}_test_score'][best] for i in range(5)]
print('CV R² per fold:', np.round(fold_scores, 4))
print(f'-> mean {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})')

# 3. Which features does the forest rely on most?
names = [n.replace('num__', '').replace('nom__', '') for n in best_rf.named_steps['preprocessor'].get_feature_names_out()]
importances = pd.Series(best_rf.named_steps['regressor'].feature_importances_, index=names).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances.index, importances.values, color='#3b6fb6', height=0.6)
for i, v in enumerate(importances.values):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature importance (share of the forest\'s total error reduction)')
ax.set_title('Random forest: which features matter most', loc='left', fontsize=13)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
### Which model is most accurate? (all scored on the same 20% test set)
results_df = (pd.DataFrame(results)
              .drop_duplicates('Model', keep='last')
              .sort_values('RMSE')
              .reset_index(drop=True))
display(results_df.style.format({'MAE': '{:.3f}', 'MSE': '{:.2f}', 'RMSE': '{:.3f}', 'R²': '{:.4f}'})
                  .highlight_min(subset=['MAE', 'MSE', 'RMSE'], color='#d4edda')
                  .highlight_max(subset=['R²'], color='#d4edda'))

best = results_df.iloc[0]
print(f"Most accurate: {best['Model']}  ->  R² {best['R²']:.4f}, RMSE {best['RMSE']:.3f} marks")

# Chart: test RMSE per model (lower = better)
plot_df = results_df.sort_values('RMSE', ascending=False)
colors = ['#2e7d32' if m == best['Model'] else '#3b6fb6' for m in plot_df['Model']]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(plot_df['Model'], plot_df['RMSE'], color=colors, height=0.6)
for bar, rmse, r2 in zip(bars, plot_df['RMSE'], plot_df['R²']):
    ax.text(rmse + 0.05, bar.get_y() + bar.get_height() / 2, f'RMSE {rmse:.3f}  ·  R² {r2:.3f}',
            va='center', fontsize=9, color='#222222')
ax.set_xlim(0, plot_df['RMSE'].max() * 1.35)
ax.set_xlabel('Test RMSE (average error in final_test marks, lower is better)')
ax.set_title('Model accuracy on the 20% test set (best in green)', loc='left', fontsize=13)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
### Double-check the linear models: are Linear, Ridge and Lasso really the same, and is anything overfitting?
from sklearn.model_selection import cross_val_score

check_models = {
    'Linear Regression': lr_pipeline,
    'Ridge (alpha=1)': ridge_pipeline,
    'Lasso (alpha=1)': lasso_pipeline,
    'Ridge + GridSearchCV': ridge_grid_search.best_estimator_,
    'Lasso + GridSearchCV': lasso_grid_search.best_estimator_,
    'Ridge + RandomizedSearchCV': ridge_random_search.best_estimator_,
    'Lasso + RandomizedSearchCV': lasso_random_search.best_estimator_,
}

# --- Check 1: train vs test score. Overfitting = much better on train than on test.
rows = []
for name, m in check_models.items():
    pred_train, pred_test = m.predict(X_train), m.predict(X_test)
    rows.append({'Model': name,
                 'Train RMSE': root_mean_squared_error(y_train, pred_train),
                 'Test RMSE': root_mean_squared_error(y_test, pred_test),
                 'Train R²': r2_score(y_train, pred_train),
                 'Test R²': r2_score(y_test, pred_test)})
gap_df = pd.DataFrame(rows)
gap_df['Gap (test - train RMSE)'] = gap_df['Test RMSE'] - gap_df['Train RMSE']
print('CHECK 1: train vs test (a large gap would mean overfitting)')
display(gap_df.style.format({c: '{:.4f}' for c in gap_df.columns if c != 'Model'}))

# --- Check 2: are the predictions / coefficients actually the same?
lr_pred = lr_pipeline.predict(X_test)
print('\nCHECK 2: how different are the predictions from Linear Regression? (in marks)')
for name, m in check_models.items():
    diff = np.abs(m.predict(X_test) - lr_pred)
    print(f'  {name:<28} max diff {diff.max():.4f} | mean diff {diff.mean():.4f}')

feature_names = lr_pipeline.named_steps['preprocessor'].get_feature_names_out()
coef_df = pd.DataFrame({name: m.named_steps['regressor'].coef_ for name, m in check_models.items()},
                       index=[f.replace('num__', '').replace('nom__', '') for f in feature_names])
print('\nCoefficients side by side (Lasso alpha=1 zeroes some out; the rest are nearly identical):')
display(coef_df.round(3))

# --- Check 3: is the score stable across 5 cross-validation folds?
scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print('\nCHECK 3: Linear Regression, 5-fold cross-validation RMSE per fold')
print('  ', np.round(-scores, 3), f'-> mean {-scores.mean():.3f} (+/- {scores.std():.3f})')

### Conclusion

- **Every model beats the baseline.** Always predicting the average score gives RMSE 13.9 marks and R² 0.00. That is the bar every model has to clear.
- **The linear models are effectively tied.** Linear Regression, Ridge, and tuned Lasso all score about **R² 0.62, RMSE 8.58 marks**, and tuning alpha makes no real difference. With about 12,200 training rows and only about 15 features there is no overfitting for regularisation to fix.
- **The random forest is far more accurate: R² 0.86, RMSE 5.2 marks.** It cuts the average error by about 40% compared with the linear models. The relationships found in the EDA are not straight lines: for example, scores peak at 5-10 study hours and drop sharply below 80% attendance. Trees capture thresholds like these, and combinations of them, directly.
- **Feature engineering still matters.** Adding class size, study-hour bands and threshold flags raised the linear models' R² from 0.589 to 0.626 when it was first tested. `class_size` is also the forest's most important feature.

## Final model choice: Random forest + RandomizedSearchCV

| Model | Test RMSE | Test MAE | Test R² |
|---|---|---|---|
| **Random forest + RandomizedSearchCV** | **5.174** | **3.624** | **0.862** |
| Random forest (default settings) | 5.603 | 3.863 | 0.839 |
| Lasso + RandomizedSearchCV (best linear model) | 8.575 | 6.804 | 0.622 |
| Linear Regression / Ridge | 8.575 | 6.804 | 0.622 |
| Lasso (alpha=1) | 10.070 | 7.970 | 0.478 |
| Baseline (always predict the mean) | 13.948 | 11.636 | 0.000 |

**We choose the random forest tuned with RandomizedSearchCV**, for three reasons:

- **It is by far the most accurate.** On unseen students it predicts `final_test` within about **3.6 marks** on average and explains about **86%** of the variation in scores. The best linear model is off by about 6.8 marks and explains 62%.
- **It is stable and doesn't overfit much.** Its training RMSE (4.98) is only about 0.2 marks better than its test RMSE (5.17). Across the 5 cross-validation folds its R² stays between 0.84 and 0.86 (mean 0.850, +/- 0.005).
- **It fits what the EDA found.** Scores depend on thresholds and peaks (for example the 5-10 study-hour peak), which a straight line can only approximate. The forest's most important features are class size, number of siblings, study hours, learning style and attendance, which match the EDA. The sleep features add almost nothing (importance about 0.001) once these are in the model, so they could be dropped in a later version.

**What we give up:** a random forest has no single coefficient per feature, so it is harder to say "one more sibling costs X marks". The linear-model coefficients above (in the double-check cell) remain a useful, easy-to-read summary of each factor's direction and size.

RandomizedSearchCV picked `n_estimators=200, max_depth=10, min_samples_leaf=2, max_features=1.0` by trying 20 combinations with 5-fold cross-validation on the training set only. Missing `attendance_rate` values are filled with the training-set median after the split. So the test score is a fair, unseen check.